# QC Prediction Of Trimming Need

        ## Manuscript targets
        - `Results / Classification of the need to trim from FastQC quality metrics`
- `QC model figures and feature-overlap discussion`

        ## Primary inputs
        - `share/results/technical/per_srr_quality.tsv`
- `share/results/technical/trimming_classification.tsv`
- `share/results/biological/qc_model_predictions.tsv`

        ## Rebuild scripts
        - `share/scripts/analysis/21_evaluate_qc_model_predictions.py`
- `share/scripts/analysis/40_retrain_qc_random_forest_model.py`

        This notebook prefers the full local `data/` working set when it exists and falls back to the tiny `share/data/` example bundle otherwise.


## Inputs, methodology, and rebuild policy

The manuscript's negative-result section asks whether FastQC features can identify the rare samples that benefit from trimming. This notebook measures baseline-vs-model accuracy, visualizes the confusion structure, and quantifies how little information the quality features contain about the beneficial-trimming label.


In [ ]:
from __future__ import annotations

import math
import re
import subprocess
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="talk")
except ImportError:
    sns = None

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 220)
plt.rcParams["figure.dpi"] = 120
SRR_PATTERN = re.compile(r"(SRR\d+)")


def find_repo_root() -> Path:
    start = Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / "share").exists() and (candidate / "share" / "results").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root from the notebook working directory.")


REPO_ROOT = find_repo_root()
SHARE = REPO_ROOT / "share"
RESULTS = SHARE / "results"
TECH = RESULTS / "technical"
BIO = RESULTS / "biological"
SUPP = RESULTS / "supplementary"
SHARE_DATA = SHARE / "data"
LOCAL_DATA = REPO_ROOT / "data"
SCRIPTS = SHARE / "scripts"
NOTEBOOK_DB = SHARE / "notebooks" / "srr_queue.db"


def data_dir(name: str) -> Path:
    local = LOCAL_DATA / name
    if local.exists():
        return local
    return SHARE_DATA / name


def load_table(path: Path) -> pd.DataFrame:
    sep = "," if path.suffix == ".csv" else "\t"
    return pd.read_csv(path, sep=sep)


def run_script(script_rel: str, *args: str) -> None:
    script = REPO_ROOT / script_rel
    if not script.exists():
        raise FileNotFoundError(script)
    cmd = ["python3", str(script), *map(str, args)] if script.suffix == ".py" else ["bash", str(script), *map(str, args)]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=REPO_ROOT)


def parse_fastqc_terminal_quality(fastqc_data_path: Path) -> float | None:
    rows: list[list[str]] = []
    in_section = False
    for line in fastqc_data_path.read_text(encoding="utf-8", errors="replace").splitlines():
        if line.startswith(">>Per base sequence quality"):
            in_section = True
            continue
        if in_section and line.startswith(">>END_MODULE"):
            break
        if in_section and line and not line.startswith("#"):
            rows.append(line.split("\t"))
    if not rows:
        return None
    means = []
    for row in rows:
        try:
            means.append(float(row[1]))
        except (IndexError, ValueError):
            continue
    if not means:
        return None
    return float(np.mean(means[-min(10, len(means)) :]))


def load_terminal_quality_table() -> pd.DataFrame:
    quality = load_table(TECH / "per_srr_quality.tsv")
    raw_dir = data_dir("flattened_fastqc_raw")
    rows = []
    if raw_dir.exists():
        for fastqc_data in raw_dir.glob("*/*_fastqc/fastqc_data.txt"):
            terminal_q = parse_fastqc_terminal_quality(fastqc_data)
            if terminal_q is None:
                continue
            project_id = fastqc_data.parents[1].name
            srr = fastqc_data.parent.name.removesuffix("_fastqc")
            if srr.endswith("_1") or srr.endswith("_2"):
                srr = srr[:-2]
            rows.append({
                "project_id": project_id,
                "SRR_ID": srr,
                "terminal_q_mean": terminal_q,
            })
    if rows:
        terminal = (
            pd.DataFrame(rows)
            .groupby(["project_id", "SRR_ID"], as_index=False)["terminal_q_mean"]
            .mean()
        )
        merged = quality.merge(terminal, on=["project_id", "SRR_ID"], how="left")
        merged["terminal_q_mean"] = merged["terminal_q_mean"].fillna(
            merged["Q_mean"] - merged["tail_quality_decay"]
        )
        return merged

    quality["terminal_q_mean"] = quality["Q_mean"] - quality["tail_quality_decay"]
    return quality


def srr_ids_from_tree(root: Path) -> set[str]:
    srrs: set[str] = set()
    if not root.exists():
        return srrs
    for path in root.rglob("*"):
        match = SRR_PATTERN.search(path.name) or SRR_PATTERN.search(path.as_posix())
        if match:
            srrs.add(match.group(1))
    return srrs


def flattened_dir_srr_sets(root: Path, cohort_projects: set[str] | None = None) -> dict[str, set[str]]:
    dir_sets: dict[str, set[str]] = {}
    if not root.exists():
        return dir_sets

    for flattened_dir in sorted(
        p for p in root.iterdir() if p.is_dir() and p.name.startswith("flattened_")
    ):
        dir_srrs: set[str] = set()
        for project_dir in sorted(p for p in flattened_dir.iterdir() if p.is_dir()):
            if cohort_projects is not None and project_dir.name not in cohort_projects:
                continue
            dir_srrs.update(srr_ids_from_tree(project_dir))
        dir_sets[flattened_dir.name] = dir_srrs
    return dir_sets


def collect_flattened_dir_audit(root: Path, cohort_projects: set[str] | None = None) -> pd.DataFrame:
    rows = []
    if not root.exists():
        return pd.DataFrame(
            columns=[
                "flattened_dir",
                "projects_total",
                "unique_srrs_total",
                "projects_in_cohort",
                "unique_srrs_in_cohort",
                "projects_outside_cohort",
                "unique_srrs_outside_cohort",
                "outside_cohort_projects",
            ]
        )

    for flattened_dir in sorted(
        p for p in root.iterdir() if p.is_dir() and p.name.startswith("flattened_")
    ):
        project_srrs: dict[str, set[str]] = {}
        for project_dir in sorted(p for p in flattened_dir.iterdir() if p.is_dir()):
            project_srrs[project_dir.name] = srr_ids_from_tree(project_dir)

        all_srrs = set().union(*project_srrs.values()) if project_srrs else set()
        if cohort_projects is None:
            in_cohort = project_srrs
            out_cohort: dict[str, set[str]] = {}
        else:
            in_cohort = {k: v for k, v in project_srrs.items() if k in cohort_projects}
            out_cohort = {k: v for k, v in project_srrs.items() if k not in cohort_projects}

        in_srrs = set().union(*in_cohort.values()) if in_cohort else set()
        out_srrs = set().union(*out_cohort.values()) if out_cohort else set()

        rows.append(
            {
                "flattened_dir": flattened_dir.name,
                "projects_total": len(project_srrs),
                "unique_srrs_total": len(all_srrs),
                "projects_in_cohort": len(in_cohort),
                "unique_srrs_in_cohort": len(in_srrs),
                "projects_outside_cohort": len(out_cohort),
                "unique_srrs_outside_cohort": len(out_srrs),
                "outside_cohort_projects": ", ".join(sorted(out_cohort)) if out_cohort else "",
            }
        )

    return pd.DataFrame(rows)


In [ ]:
quality = load_table(TECH / "per_srr_quality.tsv")
cls = load_table(TECH / "trimming_classification.tsv")
pred = load_table(BIO / "qc_model_predictions.tsv")

merged = pred.merge(
    quality,
    on=["SRR_ID", "project_id"],
    how="left",
).merge(
    cls[["SRR_ID", "project_id", "t_star"]],
    on=["SRR_ID", "project_id"],
    how="left",
)
merged["benefits_from_trimming"] = merged["t_star"].ne("U").astype(int)
merged.head()


In [ ]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import accuracy_score

baseline_accuracy = (merged["True_t_star"] == "U").mean()
model_accuracy = accuracy_score(merged["True_t_star"], merged["Predicted_t_star"])

features = [
    "sequence_depth",
    "Q_mean",
    "Q_median",
    "read_length_mean",
    "frac_below_q20",
    "frac_below_q30",
    "tail_quality_decay",
    "adapter_rate",
    "duplication_rate",
    "n_content",
    "gc_content",
    "gc_deviation",
]

mi_df = merged.dropna(subset=features + ["benefits_from_trimming"]).copy()
X = mi_df[features].to_numpy()
y = mi_df["benefits_from_trimming"].to_numpy()
mi_scores = mutual_info_classif(X, y, random_state=42, discrete_features=False)
mi_table = (
    pd.DataFrame({"feature": features, "mutual_information": mi_scores})
    .sort_values("mutual_information", ascending=False)
    .reset_index(drop=True)
)

display(
    pd.Series(
        {
            "n_prediction_rows": len(merged),
            "baseline_always_untrimmed_accuracy": baseline_accuracy,
            "random_forest_accuracy": model_accuracy,
        }
    ).to_frame("value")
)
display(mi_table)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

accuracy_df = pd.DataFrame(
    {
        "model": ["Always predict U", "Random forest"],
        "accuracy": [baseline_accuracy, model_accuracy],
    }
)
axes[0].bar(accuracy_df["model"], accuracy_df["accuracy"], color=["#9D9D9D", "#4C78A8"])
axes[0].set_ylim(0.9, 1.01)
axes[0].set_title("Prediction accuracy")
axes[0].tick_params(axis="x", rotation=20)

conf = pd.crosstab(merged["True_t_star"], merged["Predicted_t_star"])
if sns is not None:
    sns.heatmap(conf, annot=True, fmt="d", cmap="Blues", ax=axes[1])
else:
    axes[1].imshow(conf.to_numpy(), cmap="Blues")
    axes[1].set_xticks(range(len(conf.columns)))
    axes[1].set_xticklabels(conf.columns, rotation=30)
    axes[1].set_yticks(range(len(conf.index)))
    axes[1].set_yticklabels(conf.index)
axes[1].set_title("Prediction confusion matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

top_features = mi_table.head(6)
axes[2].barh(top_features["feature"], top_features["mutual_information"], color="#F58518")
axes[2].invert_yaxis()
axes[2].set_title("Top mutual-information scores")
axes[2].set_xlabel("Mutual information")

fig.tight_layout()
plt.show()


In [ ]:
top_overlap_features = mi_table.head(3)["feature"].tolist()
fig, axes = plt.subplots(1, len(top_overlap_features), figsize=(5 * len(top_overlap_features), 4), sharey=False)
if len(top_overlap_features) == 1:
    axes = [axes]
plot_df = merged.copy()
plot_df["benefit_label"] = np.where(plot_df["benefits_from_trimming"].eq(1), "beneficial", "untrimmed-optimal")
for ax, feature in zip(axes, top_overlap_features):
    if sns is not None:
        sns.boxplot(data=plot_df, x="benefit_label", y=feature, ax=ax, color="#72B7B2")
    else:
        vals = [
            plot_df.loc[plot_df["benefit_label"].eq(label), feature].dropna()
            for label in ["beneficial", "untrimmed-optimal"]
        ]
        ax.boxplot(vals, tick_labels=["beneficial", "untrimmed-optimal"])
    ax.set_title(feature)
    ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
plt.show()
